<a href="https://colab.research.google.com/github/sathkumara-smna/258739K/blob/main/site_power_cell_to_site.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# FULLY DATA-DRIVEN CELL-WISE ML MODEL
# PREDICT CELL POWER FIRST
# AGGREGATE PREDICTED CELL POWER AFTERWARD
# NO ENGINEERING EQUATIONS
# NO MANUAL POWER PARAMETERS
# OPTIMIZED / FASTER VERSION
# ============================================================

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# LOAD EXCEL FILES FROM GITHUB
# ============================================================

site_db_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/Site%20Database%20from%20Sey.xlsx"
site_power_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/site_power%20from%20Sey.xlsx"
traffic_4g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/4G%20Traffic.xlsx"
traffic_5g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/5G%20Traffic.xlsx"

# ============================================================
# READ EXCEL FILES
# ============================================================

site_db = pd.read_excel(site_db_url)
site_power = pd.read_excel(site_power_url)
traffic_4g = pd.read_excel(traffic_4g_url)
traffic_5g = pd.read_excel(traffic_5g_url)
site_db.head(2)

,#,Site_ID,Site Name,2G RRUs,3G RRUs,4G RRUs,5G AAUs,2G Boards,3G Boards,4G Boards,5G Boards,BBU 5900,BBU 3900,BBU 3910
0,1,101,AIRPORT_MAHE,3,7,12,3,1,2,2,1,1,1,1
1,2,102,AIRPORT_PRASLIN,2,4,4,0,1,1,1,0,0,1,0


In [5]:
# ============================================================
# CONVERT DATETIME
# ============================================================

site_power['datetime'] = pd.to_datetime(
    site_power['datetime']
)
traffic_4g['datetime'] = pd.to_datetime(
    traffic_4g['datetime']
)
traffic_5g['datetime'] = pd.to_datetime(
    traffic_5g['datetime']
)
site_power.head(2)

,Site_ID,trigger_ID,date,datetime,site_power
0,101,1,2026-03-01,2026-03-01 00:00:00,6517.6299
1,101,2,2026-03-01,2026-03-01 00:15:00,6533.5512


In [6]:
# ============================================================
# RENAME SITE DATABASE COLUMNS
# ============================================================

site_db.columns = [
    '#','Site_ID','Site_Name','RRU_2G','RRU_3G','RRU_4G','AAU_5G','Col_H',
    'Col_I','Boards_4G','Boards_5G','BBU5900','BBU3900','BBU3910'
]
site_db.head(2)

,#,Site_ID,Site_Name,RRU_2G,RRU_3G,RRU_4G,AAU_5G,Col_H,Col_I,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910
0,1,101,AIRPORT_MAHE,3,7,12,3,1,2,2,1,1,1,1
1,2,102,AIRPORT_PRASLIN,2,4,4,0,1,1,1,0,0,1,0


In [7]:
# ============================================================
# CREATE TIME FEATURES
# ============================================================

traffic_4g['hour'] = (
    traffic_4g['datetime'].dt.hour
)

traffic_5g['hour'] = (
    traffic_5g['datetime'].dt.hour
)
traffic_4g.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,hour
0,101,10111,11,1,2026-03-01,2026-03-01 00:00:00,11.90,0
1,101,10111,11,2,2026-03-01,2026-03-01 00:15:00,12.03,0


In [9]:
# ============================================================
# CREATE TECHNOLOGY COLUMN
# ============================================================

traffic_4g['technology'] = '4G'
traffic_5g['technology'] = '5G'
traffic_4g.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,hour,technology,lte_cell_count,...,Col_I,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910,total_4g_traffic,total_5g_traffic,site_power,cell_target_power
0,101,10111,11,1,2026-03-01,2026-03-01 00:00:00,11.90,0,4G,12,...,2,2,1,1,1,1,150.81,172.34,6517.6299,239.271312
1,101,10111,11,2,2026-03-01,2026-03-01 00:15:00,12.03,0,4G,12,...,2,2,1,1,1,1,155.48,188.49,6533.5512,227.841902


In [10]:
# ============================================================
# LTE CELL COUNTS
# ============================================================

lte_counts = (

    traffic_4g.groupby('Site_ID')['Cell_ID']
    .nunique()
    .reset_index()

)

lte_counts.rename(
    columns={'Cell_ID': 'lte_cell_count'},
    inplace=True
)
lte_counts.head(2)

,Site_ID,lte_cell_count
0,101,12
1,102,4


In [11]:
# ============================================================
# NR CELL COUNTS
# ============================================================

nr_counts = (

    traffic_5g.groupby('Site_ID')['Cell_ID']
    .nunique()
    .reset_index()

)

nr_counts.rename(
    columns={'Cell_ID': 'nr_cell_count'},
    inplace=True
)
nr_counts.head(2)

,Site_ID,nr_cell_count
0,101,3
1,103,3


In [ ]:
# ============================================================
# MERGE COUNTS TO LTE
# ============================================================

traffic_4g = traffic_4g.merge(
    lte_counts,
    on='Site_ID',
    how='left'
)

traffic_4g = traffic_4g.merge(
    nr_counts,
    on='Site_ID',
    how='left'
)
traffic_4g.head(2)

In [8]:


# ============================================================
# MERGE COUNTS TO NR
# ============================================================

traffic_5g = traffic_5g.merge(
    lte_counts,
    on='Site_ID',
    how='left'
)

traffic_5g = traffic_5g.merge(
    nr_counts,
    on='Site_ID',
    how='left'
)

# ============================================================
# MERGE SITE DATABASE
# ============================================================

traffic_4g = traffic_4g.merge(
    site_db,
    on='Site_ID',
    how='left'
)

traffic_5g = traffic_5g.merge(
    site_db,
    on='Site_ID',
    how='left'
)

# ============================================================
# AGGREGATE TOTAL TRAFFIC
# ============================================================

lte_total = (

    traffic_4g.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['traffic_load_mbps']

    .sum()

)

lte_total.rename(
    columns={'traffic_load_mbps': 'total_4g_traffic'},
    inplace=True
)

nr_total = (

    traffic_5g.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['traffic_load_mbps']

    .sum()

)

nr_total.rename(
    columns={'traffic_load_mbps': 'total_5g_traffic'},
    inplace=True
)

# ============================================================
# MERGE TOTAL TRAFFIC TO LTE
# ============================================================

traffic_4g = traffic_4g.merge(

    lte_total,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

traffic_4g = traffic_4g.merge(

    nr_total,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

# ============================================================
# MERGE TOTAL TRAFFIC TO NR
# ============================================================

traffic_5g = traffic_5g.merge(

    lte_total,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

traffic_5g = traffic_5g.merge(

    nr_total,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

# ============================================================
# MERGE ACTUAL SITE POWER
# ============================================================

traffic_4g = traffic_4g.merge(

    site_power,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

traffic_5g = traffic_5g.merge(

    site_power,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

# ============================================================
# FILL NULLS
# ============================================================

traffic_4g.fillna(0, inplace=True)

traffic_5g.fillna(0, inplace=True)

# ============================================================
# ESTIMATE CELL TARGET POWER
# ============================================================

traffic_4g['cell_target_power'] = (

    traffic_4g['site_power']

    *

    (

        traffic_4g['traffic_load_mbps']

        /

        (

            traffic_4g['total_4g_traffic']
            +
            traffic_4g['total_5g_traffic']
            +
            1

        )

    )

)

traffic_5g['cell_target_power'] = (

    traffic_5g['site_power']

    *

    (

        traffic_5g['traffic_load_mbps']

        /

        (

            traffic_5g['total_4g_traffic']
            +
            traffic_5g['total_5g_traffic']
            +
            1

        )

    )

)

# ============================================================
# COMBINE LTE + NR DATA
# ============================================================

cell_df = pd.concat(
    [traffic_4g, traffic_5g],
    ignore_index=True
)

# ============================================================
# FEATURES
# ============================================================

features = [

    'traffic_load_mbps',

    'hour',
    'trigger_ID',

    'lte_cell_count',
    'nr_cell_count',

    'RRU_2G',
    'RRU_3G',
    'RRU_4G',

    'AAU_5G',

    'Boards_4G',
    'Boards_5G',

    'BBU5900',
    'BBU3900',
    'BBU3910'

]

X = cell_df[features]

y = cell_df['cell_target_power']

# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,
    random_state=42

)

# ============================================================
# RANDOM FOREST MODEL
# ============================================================

model = RandomForestRegressor(

    n_estimators=50,
    max_depth=12,
    random_state=42,
    n_jobs=-1

)

model.fit(X_train, y_train)

# ============================================================
# PREDICT CELL POWER
# ============================================================

cell_df['predicted_cell_power'] = (

    model.predict(
        cell_df[features]
    )

)

# ============================================================
# AGGREGATE PREDICTED CELL POWER
# ============================================================

final_df = (

    cell_df.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['predicted_cell_power']

    .sum()

)

# ============================================================
# RENAME COLUMN
# ============================================================

final_df.rename(

    columns={
        'predicted_cell_power': 'predicted_site_power'
    },

    inplace=True

)

# ============================================================
# MERGE ACTUAL SITE POWER
# ============================================================

final_df = final_df.merge(

    site_power,

    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],

    how='left'

)

# ============================================================
# EVALUATION METRICS
# ============================================================

mae = mean_absolute_error(

    final_df['site_power'],
    final_df['predicted_site_power']

)

rmse = np.sqrt(

    mean_squared_error(

        final_df['site_power'],
        final_df['predicted_site_power']

    )

)

mape = np.mean(

    np.abs(

        (

            final_df['site_power']
            -
            final_df['predicted_site_power']

        )

        /

        final_df['site_power']

    )

) * 100

r2 = r2_score(

    final_df['site_power'],
    final_df['predicted_site_power']

)

# ============================================================
# PRINT RESULTS
# ============================================================

print('================================')
print('CELL-WISE ML PERFORMANCE')
print('================================')

print(f'MAE  : {round(mae, 2)}')
print(f'RMSE : {round(rmse, 2)}')
print(f'MAPE : {round(mape, 2)} %')
print(f'R2   : {round(r2, 4)}')

# ============================================================
# ERROR CALCULATION
# ============================================================

final_df['error'] = (

    final_df['site_power']
    -
    final_df['predicted_site_power']

)

final_df['error_percentage'] = (

    np.abs(final_df['error'])
    /
    final_df['site_power']

) * 100

# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance_df = pd.DataFrame({

    'Feature': features,
    'Importance': model.feature_importances_

})

importance_df = importance_df.sort_values(

    by='Importance',
    ascending=False

)

print('================================')
print('FEATURE IMPORTANCE')
print('================================')

print(importance_df)

# ============================================================
# EXPORT RESULTS
# ============================================================

final_df.to_excel(

    'Fully_Data_Driven_Cell_Wise_Predictions.xlsx',
    index=False

)

print('================================')
print('OUTPUT FILE CREATED')
print('================================')

print('Fully_Data_Driven_Cell_Wise_Predictions.xlsx')

# ============================================================
# SAMPLE RESULTS
# ============================================================

print(final_df.head(20))

KeyboardInterrupt: 

In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# ERROR PERCENTAGE
# ============================================================

final_df['error_percentage'] = (

    np.abs(

        final_df['site_power']
        -
        final_df['predicted_site_power']

    )

    /

    final_df['site_power']

) * 100

# ============================================================
# ACTUAL VS PREDICTED POWER GRAPH
# ============================================================

plt.figure(figsize=(16,6))

plt.plot(
    final_df['site_power'].values,
    label='Actual Site Power'
)

plt.plot(
    final_df['predicted_site_power'].values,
    label='Predicted Site Power'
)

plt.xlabel('Samples')
plt.ylabel('Power (W)')
plt.title('Actual vs Predicted Site Power')
plt.legend()
plt.grid(True)

plt.show()

# ============================================================
# SCATTER PLOT
# ============================================================

plt.figure(figsize=(8,8))

plt.scatter(
    final_df['site_power'],
    final_df['predicted_site_power']
)

plt.xlabel('Actual Site Power')
plt.ylabel('Predicted Site Power')
plt.title('Actual vs Predicted Scatter Plot')
plt.grid(True)
plt.show()

# ============================================================
# ERROR PERCENTAGE HISTOGRAM
# ============================================================

plt.figure(figsize=(10,6))

plt.hist(
    final_df['error_percentage'],
    bins=30
)

plt.xlabel('Error Percentage (%)')
plt.ylabel('Frequency')
plt.title('Prediction Error Distribution')
plt.grid(True)
plt.show()